# Support Vector Machines (SVMs)

In this notebook, we will be exploring Support Vector Machines (SVMs) using the existing Titanic dataset.

We will follow the basic preprocessing pipeline and train/validation split of the initial titanic.ipynb to ensure consistency.

We will then use SVM, a machine learning algorithm that learns to classify samples by finding a decision boundary, on the processed Titanic dataset. Validation accuracy of SVM will be compared to previous accuracies of logistic regression, random forest, XGBoost, and then submitted to Kaggle's Titanic competition. 

The aim for this notebook is to reinforce learning of SVMs, not for the highest score on Kaggle's Titanic competition.

## 01. Load Data

In [1]:
# Import Modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
print("Set Up Complete")

# Load Data
titanic = pd.read_csv("train.csv")
titanic.head()

Set Up Complete


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 02. Train/validation split, Basic Preprocessing Pipeline

Drop Passenger ID, Name, Ticket, Cabin columns
Separate X and y
Train/validation split
Fit preprocessing pipeline only on X_train

In [2]:
# Drop Passenger ID, Name, Ticket, Cabin
titanic_v2 = titanic.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

# Train/Test Split
from sklearn.model_selection import train_test_split
X = titanic_v2.drop(columns=["Survived"])
y = titanic_v2["Survived"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# Separate numerical and categorical columns
numerical_columns = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_columns = ["Sex", "Embarked"]

# Import necessary libraries
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Build preprocessing
numerical_transformer = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                    ("encoder", OneHotEncoder())])

# Combine both
preprocessing = ColumnTransformer([("num", numerical_transformer, numerical_columns),
                                   ("cat", categorical_transformer, categorical_columns)],
                                   remainder = 'passthrough')

## 03. SVC (Support Vector Classifier)

SVM is a machine learning algorithm then learns to separate classes by finding a decision boundary. SVMs can also handle non-linear patterns using a kernel. By default, and in this notebook, we will be using the RBF kernel. 

Numerical features will be scaled as SVMs relies on distance between datapoints, and hence are sensitive to feature scales.

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# SVC is sensitive to feature scales, so we will standardise numerical features
svc_pipeline = Pipeline([("preprocessing", preprocessing), 
                         ("scaler", StandardScaler()), 
                         ("classifier", SVC())])
svc_pipeline.fit(X_train, y_train)
svc_y_pred = svc_pipeline.predict(X_valid)
svc_accuracy = accuracy_score(y_valid, svc_y_pred)
print(f"SVC Validation Accuracy: {svc_accuracy:.4f}")

SVC Validation Accuracy: 0.8212


## 04. Submission

In [4]:
# Load test.csv
test = pd.read_csv("test.csv")

# Drop columns but keep PassengerID
test_passenger_id = test["PassengerId"]
test = test.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])


# SVC with standard scaling
svc_pipeline.fit(X_train, y_train)
svc_test_pred = svc_pipeline.predict(test)
submission = pd.DataFrame({"PassengerId": test_passenger_id, "Survived": svc_test_pred})
submission.head()
submission.to_csv("titanic_submission_v4_svm.csv", index=False)
submission.shape

(418, 2)

## 05. Conclusion

| Model | Validation Accuracy | Kaggle score |
| ----- | ------------------- | -----------: |
| Logistic Regression |0.8101 |0.76315|
| Random Forest, `n_estimators=1`|0.8268|0.67942|
| Random Forest, `n_estimators=100` |0.8045|0.75119|
| XGBoost|0.8492|**0.77751**|
| SVC| 0.8212| **0.77751**|

SVC achieved a validation accuracy of 82.12%, slightly outperforming our Logistic Regression baseline (81.01%), but underperforming our best-tuned XGBoost model (84.92%).

Yet, SVC achieved a Kaggle score of 0.77751, matching the XGBoost model's score of 0.77751. Althogh XGBoost achieved higher validation accuracy, both model performed equally on the Kaggle score. This demonstrates that different algorithms can achieve similar generalisation performance despite learning patterns differently. It also highlights validation accuracy from a single split may not reliably predict model's performance on unseen data. 

SVC learns a decision boundary that separates classes, using support vectors to determine the boundary and an RBF kernel to capture non-linear patterns. This experiment shows that different algorithms can achieve different results on the same dataset, even on the same train/validation split. Further evaluation, such as cross-validation, would help determine whether these differences in performance is consistent.